# Sentiment Regression — Overall Pipeline

In [1]:
%pip install -U -q "torchao>=0.16.0" "peft>=0.10.0" "transformers>=4.40.0" datasets accelerate evaluate scikit-learn seaborn
%load_ext autoreload
%autoreload 2

from util import *
from util import save_custom_model
from transformers import DataCollatorWithPadding

import torch
print(f"GPU Active: {torch.cuda.is_available()}")

Note: you may need to restart the kernel to use updated packages.


/home/lzurbuchen/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU Active: True


## 1. Config 
Change this to switch experiments 

In [ ]:
# ============================================================
# EXPERIMENT CONFIG
# ============================================================

# Which trainer to use: "default" | "class"
TRAINER_KIND = "class"

# Model architecture: "regressor" | "class"
MODEL_KIND = "class"

# other hyperparameters
BASE_MODEL_ID = "xlm-roberta-large"
DATA_SEED = 42           # classifier (seeded) 42. regressor 69. (classifier (all) was 41.)
ALL_DATA = True          # if you want to build a model on all training data 

LORA_R = 64              
LORA_ALPHA = LORA_R      # set to alpha = r 
LORA_DROPOUT = 0.05      

# Optimization
BATCH_SIZE = 64 
EVAL_BATCH_SIZE = 512
LEARNING_RATE = 1.5e-4   # 5e-5 for xlm-roberta-large classifier run with seeded data
WEIGHT_DECAY = 0.01      
WARMUP_STEPS = 200
NUM_EPOCHS = 2           # 1 for xlm-roberta-large run with all data  
GRAD_CLIP = 1.0          
LABEL_SMOOTHING = 0.0    

# Eval / save
EVAL_STEPS = 750         
SAVE_STEPS = 3000
EARLY_STOPPING_PATIENCE = 4  # just in case, shouldn't fire during runs

# Huber loss config
HUBER_DELTA = 0.75        

# For model checkpointing 
CHECKPOINT_DIR = f"./sentiment_results_{TRAINER_KIND}_{MODEL_KIND}_large_two"
MODELSAVE_DIR = f"./sentiment_results_model_{TRAINER_KIND}_{MODEL_KIND}_large_two"

## 2. Data loading & preprocessing
(Lang mapping pulled to a constant but otherwise same)

In [ ]:
from datasets import load_dataset, Value
from transformers import AutoTokenizer
from datasets import DatasetDict

file_path = 'data/train_lang.csv'
dataset = load_dataset('csv', data_files=file_path, split='train')

# Stratify hope: train_test_split is random here, which is fine since
# both languages are abundant. If you ever filter to one language, drop
# the stratify hint anyway — HF datasets doesn't honor it the same way.
if ALL_DATA:
    dataset = DatasetDict({"train": dataset, "test": dataset.select([])})
else:
    dataset = dataset.train_test_split(test_size=0.1, seed=DATA_SEED)

"""
# uncomment for validation from the model 
VAL_SEED = 42 # 42, 43, 44
df = pd.read_csv(file_path)
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["label"],
    random_state=VAL_SEED,
)
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "test":  Dataset.from_pandas(val_df,   preserve_index=False),
})
"""

print(f"Train: {len(dataset['train'])}  Test: {len(dataset['test'])}")

model_id = BASE_MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(model_id)

LANG_TO_ID = {"eng_Latn": 0, "deu_Latn": 1}  # explicit mapping; anything not eng is treated as "other"

def preprocess(examples):
    tokenized = tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    tokenized["label"] = [float(r) for r in examples["label"]]
    # Same 0/1 convention as before, but explicit
    tokenized["lang"] = [LANG_TO_ID.get(l, 1) for l in examples["lang"]]
    return tokenized

tokenized_ds = dataset.map(
    preprocess,
    batched=True,
    remove_columns=["sentence"],  # only drop the raw text
)
tokenized_ds = tokenized_ds.cast_column("label", Value("float32"))
tokenized_ds = tokenized_ds.cast_column("lang", Value("int64"))
tokenized_ds.set_format("torch")

Train: 252000  Test: 0


## 3. Metrics
Same as before. `rounded MAE` is the submission metric while `raw_mae` is for debugging precision.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Trainer wraps multi-output forward returns in a tuple.
    # Classifier returns {"logits", "probs"} -> for logits as (logits_arr, probs_arr).
    # Regressors return {"logits"} only -> for logits as single array.
    if isinstance(logits, tuple):
        logits = logits[0]

    if logits.ndim == 2 and logits.shape[1] == 5:
        # Classifier: argmax over class dimension (though we conduct inference differently)
        predictions = logits.argmax(axis=-1).astype(float)
    else:
        # Regressor: squeeze and clip to [0,4]
        predictions = np.clip(logits.squeeze(), 0, 4)

    preds_rounded = np.rint(predictions).astype(int)
    return {
        "mae": mean_absolute_error(labels, preds_rounded),
        "raw_mae": mean_absolute_error(labels, predictions),
    }

## 4. Models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    XLMRobertaModel, XLMRobertaConfig, XLMRobertaPreTrainedModel,
)

class XLMRobertaSentimentRegressor(nn.Module):
    """Plain regression head with sigmoid squashing to 0-4."""
    def __init__(self, model_id, config, out_dim=1):
        super().__init__()
        self.config = config
        self.roberta = XLMRobertaModel.from_pretrained(model_id)
        hidden_size = config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, 1),
        )

    def forward(self, input_ids, attention_mask, labels=None, lang=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]
        logits = self.regressor(pooled)
        prediction = 4 * torch.sigmoid(logits)
        return {"logits": prediction}


class XLMRobertaSentimentClassifier(nn.Module):
    """Classifier head with no ordinal capabilities."""
    def __init__(self, model_id, config, out_dim=1):
        super().__init__()
        self.config = config
        self.roberta = XLMRobertaModel.from_pretrained(model_id)
        hidden_size = config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, 5),
        )

    def forward(self, input_ids, attention_mask, labels=None, lang=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(pooled)                  # (B, 5) — raw logits
        probs = torch.softmax(logits, dim=-1)             # (B, 5) — for inspection
        return {"logits": logits, "probs": probs}

## 5. Trainers 

In [ ]:
from transformers import Trainer

# ---------- Plain Huber baseline ----------

class HuberTrainer(Trainer):
    """Plain trainer with Huber loss; ignores `lang`."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Drop lang if present — model doesn't use it in plain mode
        inputs = {k: v for k, v in inputs.items() if k != "lang"}
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.HuberLoss(delta=HUBER_DELTA)
        loss = loss_fct(logits.squeeze(), labels.squeeze())
        return (loss, outputs) if return_outputs else loss


# ---------- CE Loss trainer simple ----------

class CrossEntropyTrainer(Trainer):
    """Trainer with cross-entropy loss for the 5-class classifier."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        inputs = {k: v for k, v in inputs.items() if k != "lang"}
        labels = inputs.pop("labels")                     

        outputs = model(**inputs)
        logits = outputs["logits"]                        

        loss_fct = nn.CrossEntropyLoss()
        # labels must be class indices (0–4), cast in case they're float.
        loss = loss_fct(logits, labels.long())

        return (loss, outputs) if return_outputs else loss

## 6. Build the model

In [ ]:
from peft import LoraConfig, get_peft_model

if MODEL_KIND == "class":
    model = XLMRobertaSentimentClassifier(
        model_id, XLMRobertaConfig.from_pretrained(model_id)
    )
    modules_to_save = ["classifier"]
elif MODEL_KIND == "regressor":
    model = XLMRobertaSentimentRegressor(
        model_id, XLMRobertaConfig.from_pretrained(model_id)
    )
    modules_to_save = ["regressor"]
else:
    raise ValueError(f"Unknown MODEL_KIND: {MODEL_KIND}")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "query", "key", "value",
        "intermediate.dense",
        "output.dense",
    ],
    modules_to_save=modules_to_save,
    lora_dropout=LORA_DROPOUT,
    task_type="SEQ_CLS",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} || total: {total:,} || trainable%: {100 * trainable / total:.4f}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4559.48it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable: 28,968,965 || total: 589,516,810 || trainable%: 4.9140


## 7. Trainer setup

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=32 if model_id == "xlm-roberta-large" else BATCH_SIZE,
    per_device_eval_batch_size=32 if model_id == "xlm-roberta-large" else EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=NUM_EPOCHS,
    max_grad_norm=GRAD_CLIP,
    eval_strategy="no" if ALL_DATA else "steps",
    eval_steps=None if ALL_DATA else EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=not ALL_DATA,
    metric_for_best_model=None if ALL_DATA else "mae",
    greater_is_better=None if ALL_DATA else False,
    fp16=True,
    logging_steps=10,
    remove_unused_columns=False,
    report_to="none",
)

trainer_cls = {
    "default": HuberTrainer,
    "class": CrossEntropyTrainer,
}[TRAINER_KIND]

callbacks = []
if not ALL_DATA:
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE))

trainer = trainer_cls(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)

print(f"Trainer: {trainer_cls.__name__}")
print(f"Model:   {MODEL_KIND}")
print(f"LoRA:    r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")

Trainer: CrossEntropyTrainer
Model:   class
LoRA:    r=64, alpha=64, dropout=0.05


## 8. Train

In [ ]:
trainer.train()

In [11]:
model.save_pretrained(MODELSAVE_DIR)

## 9. Validation (Ensemble and Components)
Do on a different seed

In [ ]:
from peft import PeftModel

def load_lora_model(model_cls, adapter_dir, model_id="xlm-roberta-base", device="cpu"):
    # build the base model, then wrap with trained LoRA adapter
    config = XLMRobertaConfig.from_pretrained(model_id)
    base_model = model_cls(model_id=model_id, config=config)

    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.to(device).eval()
    return model

device = "cuda"

# regressor load 
reg_model = load_lora_model(
    XLMRobertaSentimentRegressor,
    "./sentiment_results_model_default_regressor_large_alldata", # changed to large for now 
    model_id = "xlm-roberta-large",
    device=device,
)

# classifier load 
cls_model = load_lora_model(
    XLMRobertaSentimentClassifier,
    "./sentiment_results_model_class_class_large_alldata",
    model_id = "xlm-roberta-large",
    device=device,
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 22510.09it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 24801.10it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias  

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm


@torch.no_grad()
def collect_predictions(model, dataloader, device):
    """Run model over a dataloader once, return raw logits + labels as numpy."""
    model.eval()
    all_logits, all_labels = [], []
    for batch in tqdm(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = out["logits"].detach().cpu().numpy()
        all_logits.append(logits)
        all_labels.append(labels.detach().cpu().numpy())
    return np.concatenate(all_logits, axis=0), np.concatenate(all_labels, axis=0)


# run once to cache raw outputs 
reg_logits, labels = collect_predictions(reg_model, trainer.get_eval_dataloader(), device)
cls_logits, _      = collect_predictions(cls_model, trainer.get_eval_dataloader(), device)

reg_scores = np.clip(reg_logits.squeeze(), 0, 4)            
cls_probs  = torch.softmax(torch.tensor(cls_logits), -1).numpy()  
cls_expected = cls_probs @ np.arange(5)                      
cls_argmax   = cls_logits.argmax(axis=-1).astype(float)      

100%|██████████| 788/788 [04:00<00:00,  3.27it/s]


'\n# FOR LR\nlr_logits = lr_model.decision_function(X_val)\nassert np.array_equal(labels, Y_val), (\n    "Row mismatch between neural eval order and LR val order. "\n    "Rebuild the trainer\'s eval dataset from dataset[\'test\'] without shuffling, "\n    "or sort both sides by a stable id column before ensembling."\n)\n\nlr_probs    = torch.softmax(torch.tensor(lr_logits), -1).numpy()      # (N, 5)\nlr_expected = lr_probs @ np.arange(5)                                 # (N,)\nlr_argmax   = lr_logits.argmax(axis=-1).astype(float)                 # (N,)\n'

In [ ]:
import pandas as pd
 
# saves validation seed to frame 
df = pd.DataFrame({
    "label": labels,
    "reg_logit": reg_logits.squeeze(),   
    "reg_score": reg_scores,             
    "cls_argmax": cls_argmax,            # hard class prediction
    "cls_expected": cls_expected,        # E[class] under softmax
})
# Raw classifier logits, one column per class
for k in range(5):
    df[f"cls_logit_{k}"] = cls_logits[:, k]
# Classifier softmax probabilities, one column per class
for k in range(5):
    df[f"cls_prob_{k}"] = cls_probs[:, k]

csv_eval_path = "eval_predictions_large.csv"
df.to_csv(csv_eval_path, index=False)
print(f"Saved {len(df)} rows to {csv_eval_path}")


Saved 25200 rows to eval_predictions.csv


In [ ]:
# load model and predictions 
df = pd.read_csv(csv_eval_path)
 
labels       = df["label"].to_numpy()
reg_logits   = df["reg_logit"].to_numpy()
reg_scores   = df["reg_score"].to_numpy()
cls_argmax   = df["cls_argmax"].to_numpy()
cls_expected = df["cls_expected"].to_numpy()
cls_logits   = df[[f"cls_logit_{k}" for k in range(5)]].to_numpy()
cls_probs    = df[[f"cls_prob_{k}"  for k in range(5)]].to_numpy()

### 9a. Ensemble-Based Experiments

In [ ]:
# blend both continuous predictions 
# regression score (continuous) + classifier's expected value (sum_k k * p_k).
def ensemble_continuous(w, reg_scores, cls_expected):
    blended = w * reg_scores + (1 - w) * cls_expected
    return np.rint(np.clip(blended, 0, 4)).astype(int)


# can change to weight search function by replacing grid with np.linspace(0, 1, 101)
def find_best_weight(predict_fn, labels, *args, grid=np.array([0.5])): 
    results = []
    for w in grid:
        preds = predict_fn(w, *args)
        mae = mean_absolute_error(labels, preds)
        results.append((w, mae))
    results = np.array(results)
    best_idx = results[:, 1].argmin()
    return results[best_idx, 0], results[best_idx, 1], results

# cls_expected is how cls is computed in inference
w1, mae1, sweep1 = find_best_weight(ensemble_continuous, labels, reg_scores, cls_expected) 
print(f"Option 1 (continuous blend): w_reg={w1:.2f}, MAE={mae1:.4f}")

# Baselines for sanity
print(f"  Reg-only MAE:  {mean_absolute_error(labels, np.rint(reg_scores).astype(int)):.4f}")
print(f"  Cls-only MAE:  {mean_absolute_error(labels, cls_argmax.astype(int)):.4f}")

Option 1 (continuous blend): w_reg=0.37, MAE=0.3547
Option 2 (rounded blend):    w_reg=0.26, MAE=0.3595
  Reg-only MAE:  0.3600
  Cls-only MAE:  0.3598


## 10. Inference and Submission 

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from tqdm.auto import tqdm
 
# run once to load test data 
df_test = pd.read_csv("data/test.csv")
test_ds = Dataset.from_pandas(df_test)
 
def preprocess_test(examples):
    return tokenizer(
        examples["sentence"], truncation=True, padding="max_length", max_length=128
    )
 
tokenized_test = test_ds.map(preprocess_test, batched=True)
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask"])
 
BATCH_SIZE = 32 if model_id == "xlm-roberta-large" else 1024
test_loader = torch.utils.data.DataLoader(tokenized_test, batch_size=BATCH_SIZE)

Map: 100%|██████████| 168000/168000 [00:28<00:00, 5926.91 examples/s]


In [ ]:
 
# predict and cache one model at a time
 
def predict_and_cache(model, kind, cache_path):
    # kind="regressor": stores reg_logit (continuous, pre-clip)
    # kind="classifier": stores 5 logit columns and 5 prob columns
    
    if os.path.exists(cache_path):
        print(f"Cache exists at {cache_path}; skipping. Delete the file to re-run.")
        return
 
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Predicting [{kind}]"):
            inputs = {k: v.to("cuda:0") for k, v in batch.items()
                      if k in ["input_ids", "attention_mask"]}
            out = model(**inputs)
            all_logits.append(out["logits"].detach().cpu().numpy())
    logits = np.concatenate(all_logits, axis=0)
 
    out_df = pd.DataFrame({"id": df_test["id"]})
 
    if kind == "regressor":
        # gives (N,) or (N, 1) continuous output 
        out_df["reg_logit"] = logits.squeeze()
 
    elif kind == "classifier":
        # gives (N, 5) raw logits 
        assert logits.ndim == 2 and logits.shape[1] == 5, f"unexpected shape {logits.shape}"
        for k in range(5):
            out_df[f"cls_logit_{k}"] = logits[:, k]
        probs = torch.softmax(torch.tensor(logits), -1).numpy()
        for k in range(5):
            out_df[f"cls_prob_{k}"] = probs[:, k]
 
    else:
        raise ValueError(f"unknown kind: {kind}")
 
    out_df.to_csv(cache_path, index=False)
    print(f"Saved {len(out_df)} rows to {cache_path}")
 
reg_test_path = "test_reg.csv"
cls_test_path = "test_cls.csv"

# call this once per model, comment others when rerunning 
predict_and_cache(reg_model, kind="regressor",  cache_path=reg_test_path)
# predict_and_cache(cls_model, kind="classifier", cache_path=cls_test_path)

Predicting [classifier]: 100%|██████████| 5250/5250 [26:22<00:00,  3.32it/s]


Saved 168000 rows to test_cls_alldata.csv


In [ ]:
# load caches, run LR, ensemble, write submission csv
 
def build_submission(weights, out_path="submission.csv"):
    """
    weights: dict mapping model name -> weight, where names must match keys below
    """
    reg_df = pd.read_csv(reg_test_path)
    cls_df = pd.read_csv(cls_test_path)
 
    # check that all caches have same id order
    assert (reg_df["id"].to_numpy() == df_test["id"].to_numpy()).all(), "reg cache misaligned"
    assert (cls_df["id"].to_numpy() == df_test["id"].to_numpy()).all(), "cls cache misaligned"
 
    # obtain continuous values per model
    reg_score = np.clip(reg_df["reg_logit"].to_numpy(), 0, 4)
    cls_probs = cls_df[[f"cls_prob_{k}" for k in range(5)]].to_numpy()
    cls_expected = cls_probs @ np.arange(5)
 
    signals = {"reg": reg_score, "cls": cls_expected}
 
    # then calculate the weighted average for ensembled classification
    total_w = sum(weights[k] for k in weights if k in signals)
    blended = sum(weights[k] * signals[k] for k in weights if k in signals) / total_w
 
    # round at the [0.5, 1.5, 2.5, ...] mark 
    final_preds = np.rint(np.clip(blended, 0, 4)).astype(int)
 
    submission = pd.DataFrame({"id": df_test["id"], "label": final_preds})
    submission.to_csv(out_path, index=False)
    print(f"Saved {out_path} with weights {weights}")
    return submission
 
 
# make submission csv 
submission_csv_path = "submission.csv"
build_submission({"reg": 0.5, "cls": 0.5},
                 out_path=submission_csv_path)

Saved submission_cls_alldata.csv with weights {'reg': 0, 'cls': 1}


,id,label
0,0,0
1,1,1
2,2,3
3,3,1
4,4,1
...,...,...
167995,167995,1
167996,167996,4
167997,167997,1
167998,167998,4
